# Docking (SMINA)



In [ ]:
import os

PDB_ID = "ID_protein"
OUT_DIR = "protein_docking_out"
EXHAUSTIVENESS = 16
NUM_MODES = 10
PADDING_A = 8.0

os.makedirs(OUT_DIR, exist_ok=True)
print("OUT_DIR:", os.path.abspath(OUT_DIR))

OUT_DIR: /Users/viktoriasazonova/Documents/ITMO/Bioinformatics/doking/hmgb1_docking_out


In [9]:
# Проверка, что smina и obabel доступны в PATH (локально)
import shutil, subprocess, sys, os, textwrap

def must_have(cmd):
    path = shutil.which(cmd)
    if not path:
        raise RuntimeError(f"Не найден '{cmd}' в PATH. Установи через conda-forge: conda install -c conda-forge {cmd}")
    return path

print("smina:", must_have("smina"))
print("obabel:", must_have("obabel"))

# версии (не обязательно, но полезно для лога)
subprocess.run(["smina", "--help"], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
print("OK: tools found")

smina: /Users/viktoriasazonova/anaconda3/envs/docking/bin/smina
obabel: /Users/viktoriasazonova/anaconda3/envs/docking/bin/obabel
OK: tools found


In [10]:
# Скачиваем PDB с RCSB
import requests, pathlib

pdb_url = f"https://files.rcsb.org/download/{PDB_ID}.pdb"
pdb_path = os.path.join(OUT_DIR, f"{PDB_ID}.pdb")

r = requests.get(pdb_url, timeout=60)
r.raise_for_status()
with open(pdb_path, "wb") as f:
    f.write(r.content)

print("Downloaded:", pdb_path, "size:", os.path.getsize(pdb_path))

Downloaded: hmgb1_docking_out/2GZK.pdb size: 316872


In [ ]:
# Удаляем ДНК/РНК (нуклеотидные остатки) из PDB: остаётся только белок
from Bio.PDB import PDBParser, PDBIO, Select

# список типичных кодов нуклеотидов в PDB
NUC_RESNAMES = {
    "DA","DT","DG","DC","DU","A","T","G","C","U","I",
    "ADE","THY","GUA","CYT","URA"
}

class ProteinOnly(Select):
    def accept_residue(self, residue):
        resname = residue.get_resname().strip()
        if resname in NUC_RESNAMES or resname == "HOH":
            return 0
        return 1

parser = PDBParser(QUIET=True)
structure = parser.get_structure(PDB_ID, pdb_path)

clean_pdb = os.path.join(OUT_DIR, f"{PDB_ID}_receptor_clean.pdb")
io = PDBIO()
io.set_structure(structure)
io.save(clean_pdb, ProteinOnly())

print("Saved receptor:", clean_pdb)

Saved receptor: hmgb1_docking_out/2GZK_receptor_clean.pdb


In [12]:
# Конвертируем receptor_clean.pdb -> receptor.pdbqt
import subprocess

receptor_pdbqt = os.path.join(OUT_DIR, f"{PDB_ID}_receptor.pdbqt")

cmd = ["obabel", clean_pdb, "-O", receptor_pdbqt, "-xh", "--partialcharge", "gasteiger"]
print("Running:", " ".join(cmd))
res = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
print(res.stdout)
if res.returncode != 0 or not os.path.exists(receptor_pdbqt):
    raise RuntimeError("OpenBabel не смог сделать receptor.pdbqt. См. лог выше.")
print("OK:", receptor_pdbqt)

Running: obabel hmgb1_docking_out/2GZK_receptor_clean.pdb -O hmgb1_docking_out/2GZK_receptor.pdbqt -xh --partialcharge gasteiger
*** Open Babel Warning  in parseAtomRecord
  Problems reading a HETATM or ATOM record.
  According to the PDB specification,
  columns 77-78 should contain the element symbol of an atom.
  but OpenBabel found 'N ' (atom 1)
*** Open Babel Warning  in parseAtomRecord
  Problems reading a HETATM or ATOM record.
  According to the PDB specification,
  columns 77-78 should contain the element symbol of an atom.
  but OpenBabel found 'C ' (atom 2)
*** Open Babel Warning  in parseAtomRecord
  Problems reading a HETATM or ATOM record.
  According to the PDB specification,
  columns 77-78 should contain the element symbol of an atom.
  but OpenBabel found 'C ' (atom 3)
*** Open Babel Warning  in parseAtomRecord
  Problems reading a HETATM or ATOM record.
  According to the PDB specification,
  columns 77-78 should contain the element symbol of an atom.
  but OpenBabel

In [ ]:
# SMILES лигандов (PubChemLite)
LIGANDS = {
    "ligand_1": "smiles_structure",
    "ligand_2": "smiles_structure",
}

In [ ]:
# Генерируем 3D-конформеры RDKit -> PDB -> PDBQT (OpenBabel)
from rdkit import Chem
from rdkit.Chem import AllChem, Descriptors

ligand_files = {}

def choose_n_confs(mol: Chem.Mol) -> int:
    heavy = mol.GetNumHeavyAtoms()
    rot = Descriptors.NumRotatableBonds(mol)
    if heavy >= 35 or rot >= 10:
        return 30
    return 1

def rdkit_build_conformers(name: str, smi: str, out_dir: str, seed: int = 42):
    mol = Chem.MolFromSmiles(smi)
    if mol is None:
        raise ValueError(f"RDKit не прочитал SMILES для {name}")
    mol = Chem.AddHs(mol)

    n_confs = choose_n_confs(mol)

    params = AllChem.ETKDGv3()
    params.randomSeed = seed
    params.pruneRmsThresh = 0.5
    params.maxAttempts = 1000

    conf_ids = AllChem.EmbedMultipleConfs(mol, numConfs=n_confs, params=params)
    if not conf_ids:
        params.randomSeed = 7
        conf_ids = AllChem.EmbedMultipleConfs(mol, numConfs=n_confs, params=params)
    if not conf_ids:
        raise RuntimeError(f"Не удалось сгенерировать 3D-конформеры для {name}.")

    # Оптимизация геометрии
    can_mmff = AllChem.MMFFHasAllMoleculeParams(mol)
    if can_mmff:
        for cid in conf_ids:
            AllChem.MMFFOptimizeMolecule(mol, mmffVariant="MMFF94s", confId=cid, maxIters=500)
    else:
        for cid in conf_ids:
            AllChem.UFFOptimizeMolecule(mol, confId=cid, maxIters=500)

    # Сохраняем каждый конформер отдельно
    pdb_paths = []
    for k, cid in enumerate(conf_ids, start=1):
        pdb_path = os.path.join(out_dir, f"{name}_conf{k:02d}.pdb")
        Chem.MolToPDBFile(mol, pdb_path, confId=cid)
        pdb_paths.append(pdb_path)

    return pdb_paths, n_confs, int(mol.GetNumHeavyAtoms()), int(Descriptors.NumRotatableBonds(mol))

def obabel_to_pdbqt(pdb_path: str, pdbqt_path: str):
    cmd = ["obabel", pdb_path, "-O", pdbqt_path, "-xh", "--partialcharge", "gasteiger"]
    res = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    if res.returncode != 0:
        print(res.stdout)
        raise RuntimeError(f"OpenBabel не смог сделать pdbqt для {os.path.basename(pdb_path)}")

for name, smi in LIGANDS.items():
    pdb_confs, n_req, heavy, rot = rdkit_build_conformers(name, smi, OUT_DIR, seed=42)

    pdbqt_confs = []
    for pdb_path in pdb_confs:
        pdbqt_path = pdb_path.replace(".pdb", ".pdbqt")
        obabel_to_pdbqt(pdb_path, pdbqt_path)
        pdbqt_confs.append(pdbqt_path)

    ligand_files[name] = {
        "pdb_confs": pdb_confs,
        "pdbqt_confs": pdbqt_confs,
        "n_confs": len(pdbqt_confs),
        "heavy_atoms": heavy,
        "rot_bonds": rot
    }

    print(f"{name}: heavy={heavy}, rot={rot}, conformers={len(pdbqt_confs)}")

ligand_files


hesperidin: heavy=43, rot=17, conformers=30
naringin: heavy=41, rot=15, conformers=26
quercetin: heavy=22, rot=6, conformers=1
psoralidin: heavy=25, rot=6, conformers=1


{'hesperidin': {'pdb_confs': ['hmgb1_docking_out/hesperidin_conf01.pdb',
   'hmgb1_docking_out/hesperidin_conf02.pdb',
   'hmgb1_docking_out/hesperidin_conf03.pdb',
   'hmgb1_docking_out/hesperidin_conf04.pdb',
   'hmgb1_docking_out/hesperidin_conf05.pdb',
   'hmgb1_docking_out/hesperidin_conf06.pdb',
   'hmgb1_docking_out/hesperidin_conf07.pdb',
   'hmgb1_docking_out/hesperidin_conf08.pdb',
   'hmgb1_docking_out/hesperidin_conf09.pdb',
   'hmgb1_docking_out/hesperidin_conf10.pdb',
   'hmgb1_docking_out/hesperidin_conf11.pdb',
   'hmgb1_docking_out/hesperidin_conf12.pdb',
   'hmgb1_docking_out/hesperidin_conf13.pdb',
   'hmgb1_docking_out/hesperidin_conf14.pdb',
   'hmgb1_docking_out/hesperidin_conf15.pdb',
   'hmgb1_docking_out/hesperidin_conf16.pdb',
   'hmgb1_docking_out/hesperidin_conf17.pdb',
   'hmgb1_docking_out/hesperidin_conf18.pdb',
   'hmgb1_docking_out/hesperidin_conf19.pdb',
   'hmgb1_docking_out/hesperidin_conf20.pdb',
   'hmgb1_docking_out/hesperidin_conf21.pdb',
   'hmg

In [15]:
# Автоматически строим blind-docking box по габаритам белка
import numpy as np
from Bio.PDB import PDBParser

def docking_box_from_pdb(pdb_path, padding=8.0):
    parser = PDBParser(QUIET=True)
    s = parser.get_structure("rec", pdb_path)
    coords = np.array([a.get_coord() for a in s.get_atoms() if a.element != "H"])
    mins = coords.min(axis=0)
    maxs = coords.max(axis=0)
    center = (mins + maxs) / 2
    size = (maxs - mins) + padding
    return center, size

center, size = docking_box_from_pdb(clean_pdb, padding=PADDING_A)
center_x, center_y, center_z = center
size_x, size_y, size_z = size

print("center:", center)
print("size  :", size)

cfg_path = os.path.join(OUT_DIR, "smina_config.txt")
with open(cfg_path, "w") as f:
    f.write(f"""center_x = {center_x:.3f}
center_y = {center_y:.3f}
center_z = {center_z:.3f}
size_x   = {size_x:.3f}
size_y   = {size_y:.3f}
size_z   = {size_z:.3f}
exhaustiveness = {EXHAUSTIVENESS}
num_modes = {NUM_MODES}
""")
print("Saved config:", cfg_path)

center: [-2.4139996 -0.9499998  3.0515003]
size  : [82.815994 52.066    55.033   ]
Saved config: hmgb1_docking_out/smina_config.txt


In [ ]:
# Запускаем SMINA для каждого лиганда:
import pandas as pd, re

def run_smina(receptor_pdbqt, ligand_pdbqt, out_pdbqt, config_path, seed=42):
    cmd = [
        "smina",
        "-r", receptor_pdbqt,
        "-l", ligand_pdbqt,
        "--config", config_path,
        "--seed", str(seed),
        "-o", out_pdbqt
    ]
    res = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    if res.returncode != 0:
        print(res.stdout)
        raise RuntimeError(f"smina упал для {os.path.basename(ligand_pdbqt)}")
    return res.stdout

def parse_best_affinity(smina_stdout):
    for line in smina_stdout.splitlines():
        if re.match(r"^\s*1\s+[-0-9.]+\s+[-0-9.]+\s+[-0-9.]+\s*$", line):
            parts = line.split()
            return float(parts[1])
    m = re.search(r"Affinity:\s*([-0-9.]+)", smina_stdout)
    if m:
        return float(m.group(1))
    return None

results = []
logs_dir = os.path.join(OUT_DIR, "logs")
os.makedirs(logs_dir, exist_ok=True)

for name, info in ligand_files.items():
    best = {"aff": None, "conf_idx": None, "pose": None, "lig_pdbqt": None}

    for i, lig_pdbqt in enumerate(info["pdbqt_confs"], start=1):
        out_pose = os.path.join(OUT_DIR, f"{name}_conf{i:02d}_docked.pdbqt")
        stdout = run_smina(receptor_pdbqt, lig_pdbqt, out_pose, cfg_path, seed=42+i)

        with open(os.path.join(logs_dir, f"{name}_conf{i:02d}.log.txt"), "w") as f:
            f.write(stdout)

        aff = parse_best_affinity(stdout)
        if aff is None:
            continue

        if (best["aff"] is None) or (aff < best["aff"]):
            best.update({"aff": aff, "conf_idx": i, "pose": out_pose, "lig_pdbqt": lig_pdbqt})

    # финальный файл
    best_pose_final = os.path.join(OUT_DIR, f"{name}_best_docked.pdbqt")
    if best["pose"] is not None and best["pose"] != best_pose_final:
        import shutil
        shutil.copyfile(best["pose"], best_pose_final)

    results.append({
        "ligand": name,
        "heavy_atoms": info["heavy_atoms"],
        "rot_bonds": info["rot_bonds"],
        "n_conformers_tested": info["n_confs"],
        "best_affinity_kcal_mol": best["aff"],
        "best_conformer_idx": best["conf_idx"],
        "best_pose_pdbqt": best_pose_final if best["pose"] is not None else None
    })

df = pd.DataFrame(results).sort_values("best_affinity_kcal_mol")
df


,ligand,heavy_atoms,rot_bonds,n_conformers_tested,best_affinity_kcal_mol,best_conformer_idx,best_pose_pdbqt
0,hesperidin,43,17,30,-10.6,9,hmgb1_docking_out/hesperidin_best_docked.pdbqt
1,naringin,41,15,26,-10.4,25,hmgb1_docking_out/naringin_best_docked.pdbqt
3,psoralidin,25,6,1,-8.6,1,hmgb1_docking_out/psoralidin_best_docked.pdbqt
2,quercetin,22,6,1,-8.4,1,hmgb1_docking_out/quercetin_best_docked.pdbqt


In [ ]:
for row in df.itertuples(index=False):
    in_pdbqt = row.best_pose_pdbqt
    out_pdb = in_pdbqt.replace(".pdbqt", ".pdb")

    cmd = ["obabel", in_pdbqt, "-O", out_pdb]
    res = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

    if res.returncode != 0:
        print(res.stdout)
        raise RuntimeError(f"Не смог конвертировать {in_pdbqt} в PDB")

    print("Saved:", out_pdb)

Saved: hmgb1_docking_out/hesperidin_best_docked.pdb
Saved: hmgb1_docking_out/naringin_best_docked.pdb
Saved: hmgb1_docking_out/psoralidin_best_docked.pdb
Saved: hmgb1_docking_out/quercetin_best_docked.pdb
